# 1. Import Libraries

In [2]:
import pandas as pd
import numpy as np

# 2. Load Cleaned Dataset

In [3]:
df = pd.read_csv("/content/ganga_water_quality_clean.csv")

df.head()

,location,year,do,bod,fecal_coliform
0,GANGA AT HARIDWAR D/S,2011,6.7,5.6,1150.0
1,GANGA AT GARHMUKTESHWAR,2011,8.2,3.4,1162.0
2,GANGA AT KANNAUJ U/S (RAJGHAT),2011,7.9,4.5,3042.0
3,"GANGA AT KANNAUJ D/S, U.P",2011,7.8,5.5,3508.0
4,GANGA AT KANPUR U/S (RANIGHAT),2011,8.6,4.3,6667.0


# 3. Check Dataset

In [4]:
df.shape

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   location        50 non-null     object 
 1   year            50 non-null     int64  
 2   do              50 non-null     float64
 3   bod             50 non-null     float64
 4   fecal_coliform  47 non-null     float64
dtypes: float64(3), int64(1), object(1)
memory usage: 2.1+ KB


# 4. Create Water Quality Indicators

In [5]:
df["do_good"] = (df["do"] > 5).astype(int)

df["bod_good"] = (df["bod"] < 3).astype(int)

df["fecal_good"] = (df["fecal_coliform"] < 2500).astype(int)

In [6]:
df.head()

,location,year,do,bod,fecal_coliform,do_good,bod_good,fecal_good
0,GANGA AT HARIDWAR D/S,2011,6.7,5.6,1150.0,1,0,1
1,GANGA AT GARHMUKTESHWAR,2011,8.2,3.4,1162.0,1,0,1
2,GANGA AT KANNAUJ U/S (RAJGHAT),2011,7.9,4.5,3042.0,1,0,0
3,"GANGA AT KANNAUJ D/S, U.P",2011,7.8,5.5,3508.0,1,0,0
4,GANGA AT KANPUR U/S (RANIGHAT),2011,8.6,4.3,6667.0,1,0,0


# 5. Create Overall Water Quality Score

In [7]:
df["water_quality_score"] = (
    df["do_good"] +
    df["bod_good"] +
    df["fecal_good"]
)

In [8]:
df[[
    "location",
    "year",
    "do_good",
    "bod_good",
    "fecal_good",
    "water_quality_score"
]].head()

,location,year,do_good,bod_good,fecal_good,water_quality_score
0,GANGA AT HARIDWAR D/S,2011,1,0,1,2
1,GANGA AT GARHMUKTESHWAR,2011,1,0,1,2
2,GANGA AT KANNAUJ U/S (RAJGHAT),2011,1,0,0,1
3,"GANGA AT KANNAUJ D/S, U.P",2011,1,0,0,1
4,GANGA AT KANPUR U/S (RANIGHAT),2011,1,0,0,1


# 6. Create Time Features

In [9]:
df["year_index"] = df["year"] - df["year"].min()

df.head()

,location,year,do,bod,fecal_coliform,do_good,bod_good,fecal_good,water_quality_score,year_index
0,GANGA AT HARIDWAR D/S,2011,6.7,5.6,1150.0,1,0,1,2,0
1,GANGA AT GARHMUKTESHWAR,2011,8.2,3.4,1162.0,1,0,1,2,0
2,GANGA AT KANNAUJ U/S (RAJGHAT),2011,7.9,4.5,3042.0,1,0,0,1,0
3,"GANGA AT KANNAUJ D/S, U.P",2011,7.8,5.5,3508.0,1,0,0,1,0
4,GANGA AT KANPUR U/S (RANIGHAT),2011,8.6,4.3,6667.0,1,0,0,1,0


# 7. Encode Monitoring Stations

In [10]:
df["station_id"] = df["location"].astype("category").cat.codes

df[["location", "station_id"]].drop_duplicates()

,location,station_id
0,GANGA AT HARIDWAR D/S,3
1,GANGA AT GARHMUKTESHWAR,2
2,GANGA AT KANNAUJ U/S (RAJGHAT),5
3,"GANGA AT KANNAUJ D/S, U.P",4
4,GANGA AT KANPUR U/S (RANIGHAT),7
5,GANGA AT KANPUR D/S (JAJMAU PUMPING STATION),6
6,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",0
7,"GANGA AT VARANASI D/S (MALVIYA BRIDGE), U.P",9
8,GANGA AT TRIGHAT (GHAZIPUR),8
9,GANGA AT DAKSHINESHWAR,1


# 8. Sort Data for Time-Series Features

In [11]:
df = df.sort_values(["location", "year"]).reset_index(drop=True)

# 9. Create Lag Features

In [12]:
df["do_lag1"] = df.groupby("location")["do"].shift(1)

In [13]:
df["bod_lag1"] = df.groupby("location")["bod"].shift(1)

In [14]:
df["fecal_coliform_lag1"] = (
    df.groupby("location")["fecal_coliform"].shift(1))

In [15]:
df[
    [
        "location",
        "year",
        "do",
        "do_lag1",
        "bod",
        "bod_lag1",
        "fecal_coliform",
        "fecal_coliform_lag1"
    ]
].head(15)

,location,year,do,do_lag1,bod,bod_lag1,fecal_coliform,fecal_coliform_lag1
0,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2011,7.2,NaN,4.0,NaN,3408.0,NaN
1,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2012,7.1,7.2,5.1,4.0,3450.0,3408.0
2,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2013,8.2,7.1,3.6,5.1,6475.0,3450.0
3,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2014,8.6,8.2,3.8,3.6,26000.0,6475.0
4,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2015,7.9,8.6,4.1,3.8,21300.0,26000.0
5,GANGA AT DAKSHINESHWAR,2011,7.8,NaN,4.0,NaN,270333.0,NaN
6,GANGA AT DAKSHINESHWAR,2012,6.1,7.8,4.1,4.0,493750.0,270333.0
7,GANGA AT DAKSHINESHWAR,2013,5.9,6.1,4.1,4.1,443333.0,493750.0
8,GANGA AT DAKSHINESHWAR,2014,6.7,5.9,4.4,4.1,592500.0,443333.0
9,GANGA AT DAKSHINESHWAR,2015,6.6,6.7,4.8,4.4,182500.0,592500.0


# 10. Create Year-to-Year Change Features

In [16]:
df["do_change"] = df["do"] - df["do_lag1"]

In [17]:
df["bod_change"] = df["bod"] - df["bod_lag1"]

In [18]:
df["fecal_coliform_change"] = (
    df["fecal_coliform"] - df["fecal_coliform_lag1"]
)

# 11. Final Feature Dataset

In [19]:
df.head()

,location,year,do,bod,fecal_coliform,do_good,bod_good,fecal_good,water_quality_score,year_index,station_id,do_lag1,bod_lag1,fecal_coliform_lag1,do_change,bod_change,fecal_coliform_change
0,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2011,7.2,4.0,3408.0,1,0,0,1,0,0,NaN,NaN,NaN,NaN,NaN,NaN
1,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2012,7.1,5.1,3450.0,1,0,0,1,1,0,7.2,4.0,3408.0,-0.1,1.1,42.0
2,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2013,8.2,3.6,6475.0,1,0,0,1,2,0,7.1,5.1,3450.0,1.1,-1.5,3025.0
3,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2014,8.6,3.8,26000.0,1,0,0,1,3,0,8.2,3.6,6475.0,0.4,0.2,19525.0
4,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2015,7.9,4.1,21300.0,1,0,0,1,4,0,8.6,3.8,26000.0,-0.7,0.3,-4700.0


In [20]:
df.shape

(50, 17)

In [21]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 17 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   location               50 non-null     object 
 1   year                   50 non-null     int64  
 2   do                     50 non-null     float64
 3   bod                    50 non-null     float64
 4   fecal_coliform         47 non-null     float64
 5   do_good                50 non-null     int64  
 6   bod_good               50 non-null     int64  
 7   fecal_good             50 non-null     int64  
 8   water_quality_score    50 non-null     int64  
 9   year_index             50 non-null     int64  
 10  station_id             50 non-null     int8   
 11  do_lag1                40 non-null     float64
 12  bod_lag1               40 non-null     float64
 13  fecal_coliform_lag1    37 non-null     float64
 14  do_change              40 non-null     float64
 15  bod_chan

In [22]:
df.isnull().sum()

,0
location,0
year,0
do,0
bod,0
fecal_coliform,3
do_good,0
bod_good,0
fecal_good,0
water_quality_score,0
year_index,0


In [23]:
model_df = df.dropna().copy()

In [24]:
model_df.shape

(36, 17)

# 12. Save Feature-Engineered Dataset

In [25]:
model_df.to_csv(
    "/content/ganga_water_quality_features.csv",
    index=False
)

In [26]:
from google.colab import files

files.download("/content/ganga_water_quality_features.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# 13. Verify Feature Dataset

In [27]:
features_df = pd.read_csv(
    "/content/ganga_water_quality_features.csv"
)

features_df.head()

,location,year,do,bod,fecal_coliform,do_good,bod_good,fecal_good,water_quality_score,year_index,station_id,do_lag1,bod_lag1,fecal_coliform_lag1,do_change,bod_change,fecal_coliform_change
0,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2012,7.1,5.1,3450.0,1,0,0,1,1,0,7.2,4.0,3408.0,-0.1,1.1,42.0
1,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2013,8.2,3.6,6475.0,1,0,0,1,2,0,7.1,5.1,3450.0,1.1,-1.5,3025.0
2,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2014,8.6,3.8,26000.0,1,0,0,1,3,0,8.2,3.6,6475.0,0.4,0.2,19525.0
3,"GANGA AT ALLAHABAD D/S (SANGAM), U.P.",2015,7.9,4.1,21300.0,1,0,0,1,4,0,8.6,3.8,26000.0,-0.7,0.3,-4700.0
4,GANGA AT DAKSHINESHWAR,2012,6.1,4.1,493750.0,1,0,0,1,1,1,7.8,4.0,270333.0,-1.7,0.1,223417.0


In [28]:
features_df.shape

(36, 17)

# Feature Engineering Summary

- Created water-quality indicators based on DO, BOD, and Fecal Coliform criteria.
- Created an overall water-quality score.
- Created a numerical year index.
- Encoded monitoring stations using station IDs.
- Created previous-year lag features for DO, BOD, and Fecal Coliform.
- Created year-to-year change features.
- Saved the feature-engineered dataset as `ganga_water_quality_features.csv`.